# Integrasi Polusi dan Kesehatan

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
BASE = ROOT / "hasil_cleaning_final"
if not (BASE / "01_aqms_clean.csv").is_file():
    raise FileNotFoundError("Jalankan cleaning.ipynb terlebih dahulu.")
OUT = ROOT / "hasil_integrasi_kesehatan"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
aqms = pd.read_csv(BASE / "01_aqms_clean.csv", parse_dates=["tanggal"])
health = pd.read_csv(BASE / "03_kesehatan_clean.csv", parse_dates=["date"])
polutan = ["pm25_ug_m3", "pm10_ug_m3", "so2_ug_m3", "co_ug_m3", "o3_ug_m3", "no2_ug_m3"]
kasus = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases", "copd_admissions"]
print("AQMS:", aqms.shape, "Kesehatan:", health.shape)


## Samakan tanggal dan wilayah

In [ ]:
def normalisasi_kota(s):
    return s.astype("string").str.strip().str.lower().str.replace(r"^kota administrasi ", "", regex=True).str.replace(r"\s+", " ", regex=True)

aqms["kota_key"] = normalisasi_kota(aqms["kota_kabupaten"])
health["kota_key"] = normalisasi_kota(health["kota_kabupaten"])
aqms["prov_key"] = normalisasi_kota(aqms["provinsi"])
health["prov_key"] = normalisasi_kota(health["provinsi"])
aqms["tanggal"] = aqms["tanggal"].dt.normalize()
health["date"] = health["date"].dt.normalize()
print("Kota AQMS:", aqms["kota_key"].nunique(), "Kota kesehatan:", health["kota_key"].nunique())


## AQMS per kota-hari

In [ ]:
kunci_sensor = ["prov_key", "kota_key", "tanggal", "sensor_id"]
sensor_harian = aqms.groupby(kunci_sensor, as_index=False)[polutan].mean()
aqms_harian = sensor_harian.groupby(["prov_key", "kota_key", "tanggal"], as_index=False)[polutan].mean()
jumlah_sensor = sensor_harian.groupby(["prov_key", "kota_key", "tanggal"], as_index=False)["sensor_id"].nunique().rename(columns={"sensor_id":"jumlah_sensor"})
aqms_harian = aqms_harian.merge(jumlah_sensor, on=["prov_key", "kota_key", "tanggal"], validate="one_to_one")
print("AQMS kota-hari:", len(aqms_harian))
display(aqms_harian.head())


## Kesehatan per kota-hari

In [ ]:
health_analisis = health.loc[health["date"] >= pd.Timestamp("2024-01-01")].copy()
health_harian = health_analisis.groupby(["prov_key", "kota_key", "date"], as_index=False)[kasus].sum(min_count=1)
health_harian = health_harian.rename(columns={"date":"tanggal"})
print("Kesehatan kota-hari:", len(health_harian))
display(health_harian.head())


In [ ]:
gabungan = health_harian.merge(aqms_harian, on=["prov_key", "kota_key", "tanggal"], how="left", validate="one_to_one", indicator=True)
print("Status merge:")
print(gabungan["_merge"].value_counts().to_string())
print("Baris gabungan:", len(gabungan), "vs kesehatan:", len(health_harian))
assert len(gabungan) == len(health_harian)
audit_merge = gabungan["_merge"].value_counts().rename_axis("status").reset_index(name="jumlah_baris")
gabungan = gabungan.drop(columns="_merge")
display(gabungan.head())


## Korelasi hari yang sama

In [ ]:
hasil = []
for p in polutan:
    for k in kasus:
        pasangan = gabungan[[p,k]].dropna()
        hasil.append({"polutan":p,"indikator":k,"lag_hari":0,"n_pasangan":len(pasangan),"korelasi_pearson":pasangan[p].corr(pasangan[k]) if len(pasangan)>=10 else float("nan")})
korelasi_0 = pd.DataFrame(hasil)
display(korelasi_0.sort_values("korelasi_pearson", key=lambda x:x.abs(), ascending=False).head(10))


## Korelasi lag 0–3 hari

In [ ]:
lag_rows=[]
for lag in range(4):
    paparan = aqms_harian[["prov_key","kota_key","tanggal"]+polutan].copy()
    paparan["tanggal"] = paparan["tanggal"] + pd.Timedelta(days=lag)
    temp = health_harian.merge(paparan, on=["prov_key","kota_key","tanggal"], how="left", validate="one_to_one")
    for p in polutan:
        for k in kasus:
            pasangan=temp[[p,k]].dropna()
            lag_rows.append({"polutan":p,"indikator":k,"lag_hari":lag,"n_pasangan":len(pasangan),"korelasi_pearson":pasangan[p].corr(pasangan[k]) if len(pasangan)>=10 else float("nan")})
korelasi_lag=pd.DataFrame(lag_rows)
display(korelasi_lag.head(12))


In [ ]:
p="pm25_ug_m3"
k="resp_admissions_under5"
plot_data=gabungan[[p,k]].dropna()
plt.figure(figsize=(7,4))
plt.scatter(plot_data[p],plot_data[k],alpha=.15,s=8)
plt.xlabel("PM2.5 (µg/m³)")
plt.ylabel("Kasus pernapasan balita (jumlah)")
plt.title("PM2.5 vs kasus balita — kota-hari, lag 0")
plt.tight_layout()
plt.savefig(OUT/"scatter_pm25_balita.png",dpi=140)
plt.show()


In [ ]:
aqms_harian.to_csv(OUT/"01_aqms_harian_kota.csv",index=False,encoding="utf-8-sig")
health_harian.to_csv(OUT/"02_kesehatan_harian_kota.csv",index=False,encoding="utf-8-sig")
gabungan.to_csv(OUT/"03_integrasi_aqms_kesehatan.csv",index=False,encoding="utf-8-sig")
korelasi_0.to_csv(OUT/"04_korelasi_lag0.csv",index=False,encoding="utf-8-sig")
korelasi_lag.to_csv(OUT/"05_korelasi_lag0_3.csv",index=False,encoding="utf-8-sig")
audit_merge.to_csv(OUT/"06_audit_merge.csv",index=False,encoding="utf-8-sig")
print("Selesai:",OUT.resolve())
